In [23]:
def parser_dimacs(caminho_arquivo):
    clausulas = []
    n_variaveis = 0
    n_clausulas = 0

    with open(caminho_arquivo, 'r') as arquivo:
        for linha in arquivo:
            # Remove espaços em branco nas extremidades
            linha = linha.strip()
            
            # 1. Ignora linhas vazias ou comentários
            if not linha or linha.startswith('c') or linha.startswith('%'):
                continue
                
            # 2. Lê o cabeçalho (p cnf n_vars n_claus)
            if linha.startswith('p'):
                partes = linha.split()
                n_variaveis = int(partes[2])
                n_clausulas = int(partes[3])
                continue
            
            # 3. Lê as cláusulas
            # Converte a linha em números inteiros
            valores = list(map(int, linha.split()))
            
            # O '0' no final serve apenas como delimitador de fim de cláusula[cite: 1]
            # Removemos o zero antes de armazenar na memória
            if valores[-1] == 0:
                valores.pop()
            
            clausulas.append(valores)

    return n_variaveis, n_clausulas, clausulas

In [24]:
# --- Exemplo de uso com o arquivo uf20-01.cnf ---
n_vars, n_claus, lista_clausulas = parser_dimacs('instances/uf20-01.cnf')
print(f"Variáveis: {n_vars}, Cláusulas: {n_claus}")
print(f"Primeira cláusula: {lista_clausulas[0]}")

Variáveis: 20, Cláusulas: 91
Primeira cláusula: [4, -18, 19]


In [ ]:
import os
import math
import random
import matplotlib.pyplot as plt

# --- 1. PARSER DIMACS ---
def parser_dimacs(caminho_arquivo):
    clausulas = []
    n_vars, n_claus = 0, 0

    with open(caminho_arquivo, 'r') as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha or linha.startswith('c'): continue
            if linha.startswith('p'):
                partes = linha.split()
            n_vars, n_claus = int(partes[2]), int(partes[3])
            continue
        #valores = list(map(int, linha.split()))
        #if valores[-1] == 0: valores.pop() # Remove o sentinela '0'
        
        valores = list(map(int, linha.split()))
        if valores and valores[-1] == 0:
            valores.pop()

        clausulas.append(valores)
        return n_vars, n_claus, clausulas

# --- 2. FUNÇÃO OBJETIVO ---
def calcular_fo(solucao, clausulas):
    satisfeitas = 0
    for clausula in clausulas:
        for literal in clausula:
            indice = abs(literal) - 1
            if (literal > 0 and solucao[indice] == 1) or (literal < 0 and solucao[indice] == 0):
                satisfeitas += 1
            break # Cláusula 'OU' satisfeita[cite: 1]
    return satisfeitas

# --- 3. GERAÇÃO DE VIZINHANÇA ---
def gerar_vizinho(solucao_atual, percentual=0.05):
    nova_solucao = list(solucao_atual)
    qtd = max(1, int(len(nova_solucao) * percentual))
    indices = random.sample(range(len(nova_solucao)), qtd)
    for idx in indices:
        nova_solucao[idx] = 1 - nova_solucao[idx]
    return nova_solucao

# --- 4. ALGORITMO SIMULATED ANNEALING ---
def simulated_annealing_3sat(clausulas, n_vars, T0, alpha, SAmax, EvalMax):
    s_atual = [random.randint(0, 1) for _ in range(n_vars)]
    fo_atual = calcular_fo(s_atual, clausulas)
    s_best, fo_best = list(s_atual), fo_atual

    T = T0
    avaliacoes = 1
    historico_fo = []

    while avaliacoes < EvalMax and T > 0.0001: # Critério de parada
        for _ in range(SAmax):
            if avaliacoes >= EvalMax: break

    s_prime = gerar_vizinho(s_atual)
    fo_prime = calcular_fo(s_prime, clausulas)
    avaliacoes += 1

    # Cálculo de Delta para Maximização
    delta = fo_atual - fo_prime # Se fo_prime > fo_atual, delta é negativo

    if delta < 0: # Melhora encontrada
        s_atual, fo_atual = list(s_prime), fo_prime
    if fo_prime > fo_best:
        s_best, fo_best = list(s_prime), fo_prime
    else:
    # Critério de Metropolis para aceitar piora
        if random.uniform(0, 1) < math.exp(-delta / T):
            s_atual, fo_atual = list(s_prime), fo_prime

    historico_fo.append(fo_atual)

    T *= alpha # Queda de temperatura

    return s_best, fo_best, historico_fo

# --- 5. EXECUÇÃO E GRÁFICO ---
if __name__ == "__main__":
    # Caminho ajustado para sua estrutura de pastas
    caminho = os.path.join(os.getcwd(), 'instances', 'uf20-01.cnf')

if os.path.exists(caminho):
    n_vars, n_claus, clausulas = parser_dimacs(caminho)

    # Parâmetros de calibragem
    params = {"T0": 100.0, "alpha": 0.99, "SAmax": 100, "EvalMax": 200000}

    sol, fo, hist = simulated_annealing_3sat(clausulas, n_vars, **params)

    print(f"Melhor FO: {fo}/{n_claus} ({(fo/n_claus)*100:.2f}%)")

    plt.plot(hist, color='blue', linewidth=0.5)
    plt.title('Convergência SA - 3-SAT')
    plt.xlabel('Avaliações'); plt.ylabel('Cláusulas Satisfeitas')
    plt.grid(True); plt.show()
else:
    print(f"Erro: Arquivo não encontrado em {caminho}")